# Gage-prioritized downstream NWM → NextGen Mapping .json

Builds `{"nex-<id>": [hf_id]}` maps straight from the VPU v2.2_hydrofabric geopackages.

For each `wb-` id in a VPU's `network` table, one `hf_id` is kept:
1. If any of its `hf_id`s is a gage (from `conus_reference.gpkg`), keep the gage with the lowest `hf_hydroseq`.
2. Otherwise, keep the `hf_id` with the lowest `hf_hydroseq`.

The output is sorted by id number (smallest to largest).

**To add a VPU:** add an entry to `JOBS` in the next cell.

In [ ]:
from pathlib import Path

HOME = Path.home()
CONUS_REF = HOME / "path/to/file/conus_reference.gpkg"
OUT_DIR = HOME / "path/to/file/"

# (VPU code, NextGen geopackage, output file)
JOBS = [
    ("01", HOME / "path/to/file/nextgen_VPU_01.gpkg", OUT_DIR / "nwm_to_ngen_map_01.json"),
    ("02", HOME / "path/to/file/nextgen_VPU_02.gpkg", OUT_DIR / "nwm_to_ngen_map_02.json"),
    ("03N", HOME / "path/to/file/nextgen_VPU_03N.gpkg", OUT_DIR / "nwm_to_ngen_map_03N.json"),
    ("03S", HOME / "path/to/file/nextgen_VPU_03S.gpkg", OUT_DIR / "nwm_to_ngen_map_03S.json"),
    ("03W", HOME / "path/to/file/nextgen_VPU_03W.gpkg", OUT_DIR / "nwm_to_ngen_map_03W.json"),
    # ("04", HOME / "path/to/file/nextgen_VPU_04.gpkg", OUT_DIR / "nwm_to_ngen_map_04.json"),
    # ("05", HOME / "path/to/file/nextgen_VPU_05.gpkg", OUT_DIR / "nwm_to_ngen_map_05.json"),
    # ("06", HOME / "path/to/file/nextgen_VPU_06.gpkg", OUT_DIR / "nwm_to_ngen_map_06.json"),
    # ("07", HOME / "path/to/file/nextgen_VPU_07.gpkg", OUT_DIR / "nwm_to_ngen_map_07.json"),
    # ("08", HOME / "path/to/file/nextgen_VPU_08.gpkg", OUT_DIR / "nwm_to_ngen_map_08.json"),
    # ("09", HOME / "path/to/file/nextgen_VPU_09.gpkg", OUT_DIR / "nwm_to_ngen_map_09.json"),
    # ("10L", HOME / "path/to/file/nextgen_VPU_10L.gpkg", OUT_DIR / "nwm_to_ngen_map_10L.json"),
    # ("10U", HOME / "path/to/file/nextgen_VPU_10U.gpkg", OUT_DIR / "nwm_to_ngen_map_10U.json"),
    # ("11", HOME / "path/to/file/nextgen_VPU_11.gpkg", OUT_DIR / "nwm_to_ngen_map_11.json"),
    # ("12", HOME / "path/to/file/nextgen_VPU_12.gpkg", OUT_DIR / "nwm_to_ngen_map_12.json"),
    # ("13", HOME / "path/to/file/nextgen_VPU_13.gpkg", OUT_DIR / "nwm_to_ngen_map_13.json"),
    # ("14", HOME / "path/to/file/nextgen_VPU_14.gpkg", OUT_DIR / "nwm_to_ngen_map_14.json"),
    # ("15", HOME / "path/to/file/nextgen_VPU_15.gpkg", OUT_DIR / "nwm_to_ngen_map_15.json"),
    # ("16", HOME / "path/to/file/nextgen_VPU_16.gpkg", OUT_DIR / "nwm_to_ngen_map_16.json"),
    # ("17", HOME / "path/to/file/nextgen_VPU_17.gpkg", OUT_DIR / "nwm_to_ngen_map_17.json"),
    # ("18", HOME / "path/to/file/nextgen_VPU_18.gpkg", OUT_DIR / "nwm_to_ngen_map_18.json")
]

In [16]:
import json, sqlite3

def load_gages(vpu):
    """Gage hf_ids for one VPU from the CONUS reference geopackage."""
    con = sqlite3.connect(CONUS_REF)
    rows = con.execute(
        "SELECT id FROM network WHERE hl_reference = 'gages' AND vpuid = ?", (vpu,)
    ).fetchall()
    con.close()
    return {r[0] for r in rows}

def build_map(vpu, gpkg, out):
    gages = load_gages(vpu)

    con = sqlite3.connect(gpkg)
    records = con.execute(
        "SELECT id, hf_id, hf_hydroseq FROM network WHERE id LIKE 'wb-%'"
    ).fetchall()
    con.close()

    best = {}
    for wb_id, hf_id, hydroseq in records:
        # Lower key wins: gages first (False < True), then lowest hydroseq
        key = (hf_id not in gages, hydroseq)
        num = int(float(wb_id.split("-")[1]))  # handles ids like "wb-5e+05"
        if num not in best or key < best[num][0]:
            best[num] = (key, hf_id)

    result = {f"nex-{n}": [best[n][1]] for n in sorted(best)}
    with open(out, "w") as f:
        json.dump(result, f, indent=4)

    n_gage = sum(not key[0] for key, _ in best.values())
    print(f"VPU {vpu}: {out}")
    return result

In [18]:
results = {vpu: build_map(vpu, gpkg, out) for vpu, gpkg, out in JOBS}

VPU 01: /home/alk05/testing/json/nwm_to_ngen_map_01.json
VPU 02: /home/alk05/testing/json/nwm_to_ngen_map_02.json
VPU 03N: /home/alk05/testing/json/nwm_to_ngen_map_03N.json
VPU 03S: /home/alk05/testing/json/nwm_to_ngen_map_03S.json
VPU 03W: /home/alk05/testing/json/nwm_to_ngen_map_03W.json
VPU 04: /home/alk05/testing/json/nwm_to_ngen_map_04.json
VPU 05: /home/alk05/testing/json/nwm_to_ngen_map_05.json
VPU 06: /home/alk05/testing/json/nwm_to_ngen_map_06.json
VPU 07: /home/alk05/testing/json/nwm_to_ngen_map_07.json
VPU 08: /home/alk05/testing/json/nwm_to_ngen_map_08.json
VPU 09: /home/alk05/testing/json/nwm_to_ngen_map_09.json
VPU 10L: /home/alk05/testing/json/nwm_to_ngen_map_10L.json
VPU 10U: /home/alk05/testing/json/nwm_to_ngen_map_10U.json
VPU 11: /home/alk05/testing/json/nwm_to_ngen_map_11.json
VPU 12: /home/alk05/testing/json/nwm_to_ngen_map_12.json
VPU 13: /home/alk05/testing/json/nwm_to_ngen_map_13.json
VPU 14: /home/alk05/testing/json/nwm_to_ngen_map_14.json
VPU 15: /home/alk05/t

In [19]:
# Quick look at the first few entries of each map
for vpu, m in results.items():
    print(vpu, list(m.items())[:5])

01 [('nex-278', [816579.0]), ('nex-279', [816401.0]), ('nex-280', [816417.0]), ('nex-281', [816383.0]), ('nex-282', [816395.0])]
02 [('nex-86372', [8084427.0]), ('nex-86373', [8084449.0]), ('nex-86374', [8084453.0]), ('nex-86375', [8084461.0]), ('nex-86376', [8084477.0])]
03N [('nex-246896', [9251194.0]), ('nex-246897', [9251350.0]), ('nex-246898', [9251512.0]), ('nex-246899', [9253266.0]), ('nex-246900', [9251532.0])]
03S [('nex-410948', [47094.0]), ('nex-410949', [21476066.0]), ('nex-410950', [21476108.0]), ('nex-410951', [21476136.0]), ('nex-410952', [21476178.0])]
03W [('nex-477846', [2040553.0]), ('nex-477847', [2042029.0]), ('nex-477848', [2040783.0]), ('nex-477849', [2040903.0]), ('nex-477850', [2040919.0])]
04 [('nex-644726', [12222822.0]), ('nex-644727', [272427.0]), ('nex-644728', [272453.0]), ('nex-644729', [272459.0]), ('nex-644730', [272603.0])]
05 [('nex-785588', [10162006.0]), ('nex-785589', [10161904.0]), ('nex-785590', [10161894.0]), ('nex-785591', [10161912.0]), ('nex